# xQTL Association Testing (cis / trans / post-processing)


## Description

This notebook runs the full xQTL association-testing workflow on the minimal working example (MWE) data: **cis-QTL** mapping, **trans-QTL** mapping, and **hierarchical multiple-testing correction**. It consumes the QC-ed genotypes produced by `2_genotype_preprocessing.ipynb`, the molecular phenotype from `1_phenotype_preprocessing.ipynb`, and the covariates (with PCs) from `4_covariates_preprocessing.ipynb`.

The example data covers chromosomes 20-22 for genotypes and chromosome 22 for the molecular phenotype (60 samples, `SAMPLE_001`..`SAMPLE_060`), so the cis/trans tests run on chr22 genes against the example variants.


## Input Files

| File | Description |
|------|-------------|
| `output/plink/protocol_example.genotype.merged.plink_qc.bed` | Merged, QC-ed genotypes (chr20-22) from `2_genotype_preprocessing.ipynb` |
| `output/rnaseq/protocol_example.rnaseq.bed.bed.gz` | Molecular phenotype (chr22 genes) from `1_phenotype_preprocessing.ipynb` |
| `output/covariate/...Marchenko_PC.gz` | Covariates incl. genotype PCs from `4_covariates_preprocessing.ipynb` |
| `reference_data/TAD/TADB_enhanced_cis.bed` | TAD-based custom cis windows |
| `data/protocol_example.combined_AD_genes.csv` | Example trans region list (chr22 genes present in the phenotype) |
| `data/look_up_gene_id.tsv` | Gene coordinate lookup for post-processing |


## Steps


### **Step 1.** Split genotypes by chromosome

cis-QTL mapping tests each gene against variants on the same chromosome, so the merged genotype is first split into one PLINK file per chromosome. This writes a `genotype_by_chrom_files.txt` list that the cis step consumes.


In [ ]:
sos run pipeline/genotype_formatting.ipynb genotype_by_chrom \
    --cwd output/genotype_by_chrom \
    --genoFile output/plink/protocol_example.genotype.merged.plink_qc.bed \
    --chrom `cut -f 1 output/plink/protocol_example.genotype.merged.plink_qc.bim | uniq | sed "s/chr//g"`

### **Step 2.** Split the molecular phenotype by chromosome (for cis)

The molecular phenotype is split per chromosome to match the per-chromosome genotypes. In this example the phenotype only contains chr22 genes, so we split on `chr22`.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_by_chrom \
    --cwd output/phenotype/phenotype_by_chrom_for_cis \
    --phenoFile output/rnaseq/protocol_example.rnaseq.bed.bed.gz \
    --name bulk_rnaseq \
    --chrom chr22

### **Step 3.** cis-QTL association (TensorQTL)

Run cis-QTL mapping with per-chromosome genotype and phenotype lists, the covariate file, and TAD-based custom cis windows. `--MAC 5` keeps variants with minor-allele count >= 5.


In [ ]:
sos run pipeline/TensorQTL.ipynb cis \
    --genotype-file output/genotype_by_chrom/protocol_example.genotype.merged.plink_qc.genotype_by_chrom_files.txt \
    --phenotype-file output/phenotype/phenotype_by_chrom_for_cis/bulk_rnaseq.phenotype_by_chrom_files.txt \
    --covariate-file `ls output/covariate/*.Marchenko_PC.gz` \
    --customized-cis-windows reference_data/TAD/TADB_enhanced_cis.bed \
    --cwd output/tensorqtl_cis/ \
    --MAC 5

In [ ]:
# Inspect cis outputs
zcat output/tensorqtl_cis/bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz | head -n 4


```
chrom   pos     molecular_trait_id      variant_id      cis_window_start_distance       cis_window_end_distance af      ma_samples      ma_count        pvalue  bhat    sebhat  molecular_trait_object
_id     n       qvalue
22      10685239        ENSG00000283047 chr22:10685239_C_T      10685239        -8274761        0.059322033     7       7       0.557003657036534       1.7881655       3.0106752       ENSG0000028304
7       59      0.5844676509015666
22      10685239        ENSG00000130538 chr22:10685239_C_T      10685239        -8274761        0.059322033     7       7       0.7893663822092614      -0.9195994      3.4118505       ENSG0000013053
8       59      0.9954985230448293
22      10685239        ENSG00000231565 chr22:10685239_C_T      10685239        -8274761        0.059322033     7       7       0.16649873339608687     -3.8369775      2.7059317       ENSG0000023156
5       59      0.9871863332304416
```

**Example output.** The `*.cis_qtl.pairs.tsv.gz` table has one row per tested SNP-gene pair, with columns including `chrom`, `pos`, `molecular_trait_id`, `variant_id`, `cis_window_start_distance`, `cis_window_end_distance`, `af`, `ma_samples`, `ma_count`, `pvalue`, `bhat`, `sebhat`. Exact values depend on the data and the covariate set, so your run will differ from any cached numbers - inspect your own output with the command above.

In [ ]:
zcat output/tensorqtl_cis/bulk_rnaseq.chr22_chr22.cis_qtl_regional_significance.tsv.gz | head -n 4

```
chrom   pos     n_variants      beta_shape1     beta_shape2     true_df p_true_df       variant_id      tss_distance    tes_distance    ma_samples      ma_count        af      p_nominal       bhat sebhat   p_perm  p_beta  molecular_trait_object_id       n_traits        genomic_inflation       q_beta  q_perm  fdr_beta        fdr_perm
22      10761227        187     0.9863207       120.65341       28.958904       0.00791524605811029     chr22:10761227_G_A      10761227        -8198773        18      18      0.15254237      0.00685575035250628   6.1449203       2.1161995       0.6185381461853815      0.6226555264551011      ENSG00000225480 1       1.5027934569218475      0.8265754178815791      0.8353626804355407      0.9480206792030608    0.9511279087918546
22      12194389        257     0.998542        175.55527       29.148886       0.001705955582317       chr22:12194389_G_A      12194389        -8165611        7       7       0.059322033     0.00145567396123194   -13.005229      3.7100265       0.2586741325867413      0.259636083996807       ENSG00000273643 1       1.3546756620994034      0.742215270659912       0.7524178563926073      0.851265849169859     0.8566885246959174
22      15574750        257     0.98622334      166.05998       28.786993       0.00671986043375028     chr22:15574750_C_A      15574750        -4785250        10      10      0.0862069       0.00564218150064473   8.107173        2.7188876       0.6781321867813219      0.6791799847027058      ENSG00000278110 1       1.1383816913541325      0.8399635424782285      0.8448145505623267      0.963375864826533     0.961889626640173
```

**Example output.** The `*.cis_qtl_regional_significance.tsv.gz` table has one row per gene (molecular trait) with permutation-based regional statistics, including columns such as `chrom`, `pos`, `n_variants`, `beta_shape1`, `beta_shape2`, `true_df`, `p_true_df`, `variant_id`, `tss_distance`, `tes_distance`, `ma_samples`, `ma_count`, `af`, `p_nominal`, `bhat`, `sebhat`, `p_perm`, `p_beta`, `q_beta`, `q_perm`. Values are data-dependent and will vary by run.

### **Step 4.** Split the molecular phenotype by chromosome (for trans)

trans-QTL mapping uses a separate per-chromosome phenotype set. We generate it the same way as for cis, writing into a distinct directory.


In [ ]:
sos run pipeline/phenotype_formatting.ipynb phenotype_by_chrom \
    --cwd output/phenotype/phenotype_by_chrom_for_trans \
    --phenoFile output/rnaseq/protocol_example.rnaseq.bed.bed.gz \
    --name bulk_rnaseq \
    --chrom chr22

### **Step 5.** trans-QTL association (TensorQTL)

trans-QTL mapping tests selected genes (given by `--region-list`) against variants genome-wide. `data/protocol_example.genotype_trans_files.txt` lists the per-chromosome genotypes, and `data/protocol_example.combined_AD_genes.csv` lists example chr22 genes (gene id in column 4).


> **Note on toy data:** The trans-QTL scan runs successfully but may report *"No trans-QTLs found across any chromosome combinations"*. This is expected for this minimal example given the small number of samples and variants — it is not an error. The pipeline step itself completes normally.


In [ ]:
sos run pipeline/TensorQTL.ipynb trans \
    --genotype-file data/protocol_example.genotype_trans_files.txt \
    --phenotype-file output/phenotype/phenotype_by_chrom_for_trans/bulk_rnaseq.phenotype_by_chrom_files.txt \
    --region-list data/protocol_example.combined_AD_genes.csv \
    --region-list-phenotype-column 4 \
    --covariate-file `ls output/covariate/*.Marchenko_PC.gz` \
    --cwd output/tensorqtl_trans/ \
    --MAC 5

In [ ]:
# Inspect trans output
zcat output/tensorqtl_trans/bulk_rnaseq.chr22_chr22.trans_qtl.pairs.tsv.gz | head -n 4

No trans-QTLs found across any chromosome combinations in our example.

**Note on toy data**: The trans-QTL scan runs successfully but may report "No trans-QTLs found across any chromosome combinations". This is expected for this minimal example given the small number of samples and variants, it is not an error. The pipeline step itself completes normally.

### **Step 6.** Hierarchical multiple-testing correction (post-processing)

The cis nominal results are corrected with a hierarchical strategy: local (within-gene) adjustment, then global (across-gene) FDR / q-value control, then significant-SNP selection. `--gene-coordinates` provides the gene lookup. A relaxed `--fdr_threshold 0.9` is used here so the output structure is visible even though the small example data has no real signal.


In [ ]:
sos run pipeline/qtl_association_postprocessing.ipynb default \
    --cwd output/tensorqtl_cis \
    --gene-coordinates data/look_up_gene_id.tsv \
    --sub-dir . \
    --tss_dist_col cis_window_start_distance \
    --tes_dist_col cis_window_end_distance \
    --pecotmr-path pecotmr \
    --maf-cutoff 0.01 --cis-window 1000000 \
    --regional-pattern "*.cis_qtl.regional.tsv.gz$" \
    --output-dir output/heirarchical_multi_test/output \
    --archive-dir output/heirarchical_multi_test/archive \
    --fdr_threshold 0.9

In [ ]:
# Inspect the consolidated multiple-testing result and per-gene summary
ls output/heirarchical_multi_test/output/tensorqtl_cis/

```
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_pairs.significant_qtl.filtered_bonferroni_BH_adjusted.tsv.gz
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_pairs.significant_qtl.original_bonferroni_BH_adjusted.tsv.gz
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_pairs.significant_qtl.permutation_adjusted.tsv.gz
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_pairs.significant_qtl.q_beta_adjusted_events_qvalue.tsv.gz
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_regional.significant_events.filtered_bonferroni_BH_adjusted.tsv.gz
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_regional.significant_events.original_bonferroni_BH_adjusted.tsv.gz
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_regional.significant_events.permutation_adjusted.tsv.gz
bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz.cis_regional.summary.txt
tensorqtl_cis_multiple_testing_consolidated.rds
```

In [ ]:
head -n 5 output/heirarchical_multi_test/output/tensorqtl_cis/*.cis_regional.summary.txt

```
statistic       value
permutation_fdr_perm_0.90       62
permutation_q_perm_0.90 200
permutation_fdr_perm_0.05       0
permutation_q_perm_0.05 0
```

**Example output.** The `*.cis_regional.summary.txt` reports per-threshold counts of significant events, e.g. `permutation_fdr_perm_0.90`, `permutation_q_perm_0.90`, `permutation_fdr_perm_0.05`, `permutation_q_perm_0.05`. With the relaxed `--fdr_threshold 0.9` used here the relaxed thresholds report nonzero counts while the stringent (0.05) ones are typically 0 on this small toy dataset. The exact counts are data-dependent - inspect your own summary.

## Output Files

| File | Description |
|------|-------------|
| `output/tensorqtl_cis/bulk_rnaseq.chr22_chr22.cis_qtl.pairs.tsv.gz` | cis SNP-gene nominal associations |
| `output/tensorqtl_cis/bulk_rnaseq.chr22_chr22.cis_qtl_regional_significance.tsv.gz` | Per-gene cis regional significance |
| `output/tensorqtl_trans/bulk_rnaseq.chr22_chr22.trans_qtl.pairs.tsv.gz` | trans SNP-gene associations |
| `output/heirarchical_multi_test/output/tensorqtl_cis/tensorqtl_cis_multiple_testing_consolidated.rds` | Consolidated multiple-testing-corrected results |
| `output/heirarchical_multi_test/output/tensorqtl_cis/*.cis_regional.summary.txt` | Per-gene summary of significant events/QTLs |


## Anticipated Results

Step 3 produces cis SNP-gene pairs and per-gene regional significance for chr22 genes. Step 5 produces trans SNP-gene associations for the selected example genes. Step 6 applies hierarchical correction and writes a consolidated RDS plus a per-gene summary; with the relaxed threshold you can inspect the event-level and QTL-level output structure even on this small example.
